# Chargement et visualisation des données

In [116]:
import pandas as pd

In [117]:
data_path = "../data/fra_perfumes.csv"

df = pd.read_csv(data_path)
df.head()

,Name,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url
0,9am Afnanfor women,for women,3.73,174,"['citrus', 'musky', 'woody', 'aromatic', 'warm...",[],9ambyAfnanis a fragrance for women. Top notes ...,https://www.fragrantica.com/perfume/Afnan/9am-...
1,9am Dive Afnanfor women and men,for women and men,4.29,842,"['fruity', 'woody', 'green', 'warm spicy', 'ar...",[],9am DivebyAfnanis a Aromatic Aquatic fragrance...,https://www.fragrantica.com/perfume/Afnan/9am-...
2,9am pour Femme Afnanfor women,for women,4.00,68,"['fruity', 'musky', 'amber', 'citrus', 'powder...",[],9am pour FemmebyAfnanis a Amber fragrance for ...,https://www.fragrantica.com/perfume/Afnan/9am-...
3,9pm Afnanfor men,for men,4.50,"6,865","['vanilla', 'amber', 'warm spicy', 'cinnamon',...",[],9pmbyAfnanis a Amber Vanilla fragrance for men...,https://www.fragrantica.com/perfume/Afnan/9pm-...
4,9pm pour Femme Afnanfor women,for women,3.49,63,"['woody', 'aromatic', 'rose', 'fruity', 'powde...",[],9pm pour FemmebyAfnanis a Amber Floral fragran...,https://www.fragrantica.com/perfume/Afnan/9pm-...


In [118]:
df["Description"].head()

0    9ambyAfnanis a fragrance for women. Top notes ...
1    9am DivebyAfnanis a Aromatic Aquatic fragrance...
2    9am pour FemmebyAfnanis a Amber fragrance for ...
3    9pmbyAfnanis a Amber Vanilla fragrance for men...
4    9pm pour FemmebyAfnanis a Amber Floral fragran...
Name: Description, dtype: object

In [119]:
# Afficher toutes les valeurs de la colonne Perfumers
df["Perfumers"].unique()

array(['[]', "['Christian Carbonnel']", "['Jérôme Epinette']", ...,
       "['Francisco Carbonnel']", "['Eugène Frezzati', 'Georges Würz']",
       "['Francis Bocris']"], dtype=object)

In [120]:
df.shape

(70103, 8)

In [121]:
#Retire les doublons 
df.drop_duplicates(inplace=True)
df.shape

(69991, 8)

__There were duplicates in the data. We need to handle those in the main program.__

In [122]:
# Affiche le compte des valeurs nulles par colonne
print(df.isnull().sum())

Name               3
Gender             3
Rating Value    6177
Rating Count    6177
Main Accords       0
Perfumers          0
Description        3
url                0
dtype: int64


In [123]:
missing_names = df[df['Name'].isnull()]

# Affichage
print(missing_names)

      Name Gender  Rating Value Rating Count Main Accords Perfumers  \
69527  NaN    NaN           NaN          NaN           []        []   
69708  NaN    NaN           NaN          NaN           []        []   
69745  NaN    NaN           NaN          NaN           []        []   

      Description                                                url  
69527         NaN  https://www.fragrantica.com/perfume/Calvin-Kle...  
69708         NaN  https://www.fragrantica.com/perfume/Trump/Vict...  
69745         NaN  https://www.fragrantica.com/perfume/Trump/Vict...  


## Suppression des lignes vides (les lignes avec seulement l'url compte comme vides)

In [124]:
import numpy as np

df.replace(r'^\s*$|^\[\]$', np.nan, regex=True, inplace=True)

# 2. Maintenant que tout est converti en "vrais" NaN, on supprime
df.dropna(how='all', inplace=True)

In [125]:
missing_names = df[df['Name'].isnull()]
print(missing_names)

      Name Gender  Rating Value Rating Count Main Accords Perfumers  \
69527  NaN    NaN           NaN          NaN          NaN       NaN   
69708  NaN    NaN           NaN          NaN          NaN       NaN   
69745  NaN    NaN           NaN          NaN          NaN       NaN   

      Description                                                url  
69527         NaN  https://www.fragrantica.com/perfume/Calvin-Kle...  
69708         NaN  https://www.fragrantica.com/perfume/Trump/Vict...  
69745         NaN  https://www.fragrantica.com/perfume/Trump/Vict...  


## Conversion en numérique des colonnes liées au Rating

In [126]:
# Check Rating Value and Rating Count types
print(df['Rating Value'].dtype)
print(df['Rating Count'].dtype)

float64
object


In [127]:
df["rating_value"] = pd.to_numeric(df["Rating Value"], errors="coerce")
df["rating_count"] = pd.to_numeric(df["Rating Count"], errors="coerce")
df["rating_count"] = df["rating_count"].fillna(0).astype(int)

In [128]:
print(df['rating_value'].dtype)
print(df['rating_count'].dtype)

float64
int64


Les données sont bien séparées mais il est reste des actions à effectuer : 
- Nettoyer la colonne "Name" pour retirer le "for women"/"for men"/"for women and men" (information redindante puisque l'on a la colonne "Gender")
- Remplacer "for women"/"for men"/"for women and men" par men, women, unisex
- Séparer les mains accords 
- Séparer les "Perfumers" quand on a l'information
- Extraire les notes de la description

## Nettoyage du nom

In [129]:
def clean_names(df):
    df["Name"] = df["Name"].str.strip()

    #Replace "for women", "for men", and "for women and men" from the name of the perfume
    df["Name"] = df["Name"].str.replace("for women and men", "").str.replace("for women", "").str.replace("for men", "")

    print("Names cleaned.")
    return df

df = clean_names(df)
df.head()

Names cleaned.


,Name,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url,rating_value,rating_count
0,9am Afnan,for women,3.73,174,"['citrus', 'musky', 'woody', 'aromatic', 'warm...",NaN,9ambyAfnanis a fragrance for women. Top notes ...,https://www.fragrantica.com/perfume/Afnan/9am-...,3.73,174
1,9am Dive Afnan,for women and men,4.29,842,"['fruity', 'woody', 'green', 'warm spicy', 'ar...",NaN,9am DivebyAfnanis a Aromatic Aquatic fragrance...,https://www.fragrantica.com/perfume/Afnan/9am-...,4.29,842
2,9am pour Femme Afnan,for women,4.00,68,"['fruity', 'musky', 'amber', 'citrus', 'powder...",NaN,9am pour FemmebyAfnanis a Amber fragrance for ...,https://www.fragrantica.com/perfume/Afnan/9am-...,4.00,68
3,9pm Afnan,for men,4.50,"6,865","['vanilla', 'amber', 'warm spicy', 'cinnamon',...",NaN,9pmbyAfnanis a Amber Vanilla fragrance for men...,https://www.fragrantica.com/perfume/Afnan/9pm-...,4.50,0
4,9pm pour Femme Afnan,for women,3.49,63,"['woody', 'aromatic', 'rose', 'fruity', 'powde...",NaN,9pm pour FemmebyAfnanis a Amber Floral fragran...,https://www.fragrantica.com/perfume/Afnan/9pm-...,3.49,63


## Simplification des labels de genre

In [130]:
def replace_gender(gender):
    "Replace the gender column with men, women, unisex"

    if gender == "for women":
        return "women"
    elif gender == "for men":
        return "men"
    elif gender == "for women and men":
        return "unisex"
    else:
        return gender

In [131]:
df["Gender"].unique()

array(['for women', 'for women and men', 'for men', nan], dtype=object)

In [132]:
df["Gender"]= df["Gender"].apply(replace_gender)
df["Gender"].unique()

array(['women', 'unisex', 'men', nan], dtype=object)

## Extraction des types de notes de la description

In [133]:
description = df.iloc[0]['Description']

In [134]:
print(description)

9ambyAfnanis a fragrance for women. Top notes are Lemon, Mandarin Orange, Cardamom and Pink Pepper; middle notes are Lavender, Green Apple, Orange Blossom and Rose; base notes are Musk, Moss, Cedar and Patchouli.


In [135]:
import re

In [136]:
def extract_olfactory_notes(description):
    if pd.isna(description) or not isinstance(description, str):
        return {'top': None, 'middle': None, 'base': None}

    pattern = r"Top notes are (?P<top>.*?); middle notes are (?P<middle>.*?); base notes are (?P<base>.*)\."
    
    match = re.search(pattern, description)
    
    if match:
        return {
            'top': match.group('top'),
            'middle': match.group('middle'),
            'base': match.group('base')
        }
    return {'top': None, 'middle': None, 'base': None}

def clean_olfactory_notes(note_string):
    if not note_string: return []
    standardized = note_string.replace(' and ', ', ')
    return [n.strip() for n in standardized.split(',')]

def extract_notes_from_description(df):
    df['Top Notes'] = df['Description'].apply(lambda x: extract_olfactory_notes(x)['top'])
    df['Middle Notes'] = df['Description'].apply(lambda x: extract_olfactory_notes(x)['middle'])
    df['Base Notes'] = df['Description'].apply(lambda x: extract_olfactory_notes(x)['base'])
    df['Top Notes'] = df['Top Notes'].apply(clean_olfactory_notes)
    df['Middle Notes'] = df['Middle Notes'].apply(clean_olfactory_notes)
    df['Base Notes'] = df['Base Notes'].apply(clean_olfactory_notes)
    return df

In [137]:
df = extract_notes_from_description(df)
df.head()

,Name,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url,rating_value,rating_count,Top Notes,Middle Notes,Base Notes
0,9am Afnan,women,3.73,174,"['citrus', 'musky', 'woody', 'aromatic', 'warm...",NaN,9ambyAfnanis a fragrance for women. Top notes ...,https://www.fragrantica.com/perfume/Afnan/9am-...,3.73,174,"[Lemon, Mandarin Orange, Cardamom, Pink Pepper]","[Lavender, Green Apple, Orange Blossom, Rose]","[Musk, Moss, Cedar, Patchouli]"
1,9am Dive Afnan,unisex,4.29,842,"['fruity', 'woody', 'green', 'warm spicy', 'ar...",NaN,9am DivebyAfnanis a Aromatic Aquatic fragrance...,https://www.fragrantica.com/perfume/Afnan/9am-...,4.29,842,"[Lemon, Mint, Black Currant, Pink Pepper]","[Apple, Incense, Cedar]","[Ginger, Sandalwood, Patchouli, Jasmine]"
2,9am pour Femme Afnan,women,4.00,68,"['fruity', 'musky', 'amber', 'citrus', 'powder...",NaN,9am pour FemmebyAfnanis a Amber fragrance for ...,https://www.fragrantica.com/perfume/Afnan/9am-...,4.00,68,"[Mandarin Orange, Grapefruit, Bergamot]","[Raspberry, Black Currant]","[Musk, Amber, Orange]"
3,9pm Afnan,men,4.50,"6,865","['vanilla', 'amber', 'warm spicy', 'cinnamon',...",NaN,9pmbyAfnanis a Amber Vanilla fragrance for men...,https://www.fragrantica.com/perfume/Afnan/9pm-...,4.50,0,"[Apple, Cinnamon, Wild Lavender, Bergamot]","[Orange Blossom, Lily-of-the-Valley]","[Vanilla, Tonka Bean, Amber, Patchouli]"
4,9pm pour Femme Afnan,women,3.49,63,"['woody', 'aromatic', 'rose', 'fruity', 'powde...",NaN,9pm pour FemmebyAfnanis a Amber Floral fragran...,https://www.fragrantica.com/perfume/Afnan/9pm-...,3.49,63,"[Raspberry, Violet, Apple, Orange]","[Rose, Iris, Peony, Jasmine]","[Cypress, Pine, Cedar, Amber]"


## On transforme les Accords en liste pour pouvoir itérer dessus ensuite

In [138]:
#Type de la colonne Main Accords
print(df['Main Accords'].dtype)

object


In [139]:
import ast
""""
def clean_main_accords(accord_value):
    if pd.isna(accord_value):
        return []
    if isinstance(accord_value, list):
        return [str(a).strip() for a in accord_value if a is not None]
    if isinstance(accord_value, str):
        value = accord_value.strip()
        if value.startswith('[') and value.endswith(']'):
            try:
                parsed = ast.literal_eval(value)
                if isinstance(parsed, list):
                    return [str(a).strip() for a in parsed if a is not None]
            except (ValueError, SyntaxError):
                pass
        value = value.replace(' and ', ', ')
        return [item.strip() for item in value.split(',') if item.strip()]
    return []

df["Main Accords"] = df["Main Accords"].apply(clean_main_accords)
df.head()
"""


'"\ndef clean_main_accords(accord_value):\n    if pd.isna(accord_value):\n        return []\n    if isinstance(accord_value, list):\n        return [str(a).strip() for a in accord_value if a is not None]\n    if isinstance(accord_value, str):\n        value = accord_value.strip()\n        if value.startswith(\'[\') and value.endswith(\']\'):\n            try:\n                parsed = ast.literal_eval(value)\n                if isinstance(parsed, list):\n                    return [str(a).strip() for a in parsed if a is not None]\n            except (ValueError, SyntaxError):\n                pass\n        value = value.replace(\' and \', \', \')\n        return [item.strip() for item in value.split(\',\') if item.strip()]\n    return []\n\ndf["Main Accords"] = df["Main Accords"].apply(clean_main_accords)\ndf.head()\n'

In [140]:
""""
def clean_main_accords_dataset(df):
    df["Main Accords"] = df["Main Accords"].apply(clean_main_accords)
    return df
"""

'"\ndef clean_main_accords_dataset(df):\n    df["Main Accords"] = df["Main Accords"].apply(clean_main_accords)\n    return df\n'

In [141]:
def parse_string_to_list(value):
    """Transforme une valeur (NaN, liste, string ou string-list) en une vraie liste Python."""
    if pd.isna(value):
        return []
    
    if isinstance(value, list):
        return [str(a).strip() for a in value if a is not None]
    
    if isinstance(value, str):
        val = value.strip()
        # Gestion du format list-string "['Item 1', 'Item 2']"
        if val.startswith('[') and val.endswith(']'):
            try:
                parsed = ast.literal_eval(val)
                if isinstance(parsed, list):
                    return [str(a).strip() for a in parsed if a is not None]
            except (ValueError, SyntaxError):
                pass
        
        # Gestion du texte brut "Name 1 and Name 2" ou "Name 1, Name 2"
        val = val.replace(' and ', ', ')
        return [item.strip() for item in val.split(',') if item.strip()]
    
    return []

In [142]:
def clean_perfumers_dataset(df):
    df["Perfumers"] = df["Perfumers"].apply(parse_string_to_list)
    return df

def clean_main_accords_dataset(df):
    df["Main Accords"] = df["Main Accords"].apply(parse_string_to_list)
    return df

In [143]:
df = clean_main_accords_dataset(df)
df = clean_perfumers_dataset(df)

In [147]:
# Filtre : 'notna()' pour les vrais vides ET != '[]' pour les listes vides en texte
top_10_perfumers = df[df['Perfumers'].notna() & (df['Perfumers'].astype(str) != "[]")].head(10)

# Affichage
top_10_perfumers

,Name,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url,rating_value,rating_count,Top Notes,Middle Notes,Base Notes,launch_year
137,Hayati Al Haramain Perfumes,unisex,4.21,136,"[musky, oud, powdery, woody, amber, rose, swee...",[Christian Carbonnel],HayatibyAl Haramain Perfumesis a Amber Floral ...,https://www.fragrantica.com/perfume/Al-Haramai...,4.21,136,"[Musk, Amber]","[Musk, Rose, Sugar]","[Musk, Agarwood (Oud), Woody Notes]",NaN
140,Junoon Al Haramain Perfumes,women,3.99,220,"[powdery, vanilla, iris, sweet]",[Christian Carbonnel],JunoonbyAl Haramain Perfumesis a Floral fragra...,https://www.fragrantica.com/perfume/Al-Haramai...,3.99,220,"[Powdery Notes, Rose, Ylang-Ylang, Jasmine]","[Iris, Tonka Bean, Rose, Jasmine]","[Heliotrope, Vanilla, Musk]",2016.0
200,Urbanist Homme Al Haramain Perfumes,men,3.84,19,"[aromatic, citrus, woody, fresh spicy, amber, ...",[Christian Carbonnel],Urbanist HommebyAl Haramain Perfumesis a Amber...,https://www.fragrantica.com/perfume/Al-Haramai...,3.84,19,"[Petitgrain, Orange, Lemon]","[Lavender, Rosemary, Jasmine]","[Amber, Musk, Sandalwood, Tonka Bean]",NaN
441,God Is A Woman Ariana Grande,women,3.89,"2,470","[fruity, sweet, powdery, vanilla, aquatic, mus...",[Jérôme Epinette],God Is A WomanbyAriana Grandeis a Amber Floral...,https://www.fragrantica.com/perfume/Ariana-Gra...,3.89,0,"[Pear, Ambrette (Musk Mallow)]","[Orris, Turkish Rose]","[Madagascar Vanilla, Sandalwood]",2021.0
443,R.E.M. Ariana Grande,women,3.91,"2,714","[sweet, lavender, powdery, caramel, fruity, sa...","[Clement Gavarry, Dora Baghriche]",R.E.M.byAriana Grandeis a Amber Vanilla fragra...,https://www.fragrantica.com/perfume/Ariana-Gra...,3.91,0,"[Zefir, Caramel, Salt, Fig, Quince]","[Lavender, Pear Blossom]","[Musk, Tonka Bean, Sandalwood]",2020.0
448,God Is A Woman Body Mist Ariana Grande,women,4.00,9,"[vanilla, fruity, sweet, powdery, floral, musk...",[Jérôme Epinette],God Is A Woman Body MistbyAriana Grandeis a Am...,https://www.fragrantica.com/perfume/Ariana-Gra...,4.00,9,"[Pear, Musk Mallow]","[Turkish Rose, Orris]","[Madagascar Vanilla, Sandalwood]",NaN
452,"Thank U, Next Body Mist Ariana Grande",women,4.04,23,"[sweet, coconut, fruity, vanilla, almond, musky]",[Jérôme Epinette],"Thank U, Next Body MistbyAriana Grandeis a Flo...",https://www.fragrantica.com/perfume/Ariana-Gra...,4.04,23,"[Raspberry, Pear]","[Coconut, Pink Rose]","[Macarons, Musk]",NaN
453,Cloud Ariana Grande,women,3.99,"12,730","[sweet, lactonic, vanilla, coconut, musky]",[Clement Gavarry],CloudbyAriana Grandeis a Floral Fruity Gourman...,https://www.fragrantica.com/perfume/Ariana-Gra...,3.99,0,"[Lavender, Pear, Bergamot]","[Whipped Cream, Coconut, Praline, Vanilla Orchid]","[Musk, Woody Notes]",2018.0
455,Cloud Pink Ariana Grande,women,4.04,"1,043","[sweet, fruity, tropical, musky, floral, woody...",[Clement Gavarry],Cloud PinkbyAriana Grandeis a Amber Vanilla fr...,https://www.fragrantica.com/perfume/Ariana-Gra...,4.04,0,"[Pitahaya, Wild Berries, Pineapple]","[Coconut Water, Vanilla Orchid, Ambrette]","[Praline, Musk, Amberwood, Moss]",2023.0
456,Mod Blush Ariana Grande,women,3.82,806,"[fruity, sweet, musky, rose, floral, citrus, t...",[Frank Voelkl],Mod BlushbyAriana Grandeis a fragrance for wom...,https://www.fragrantica.com/perfume/Ariana-Gra...,3.82,806,"[Raspberry, Passionfruit, Pink Pepper, Bergamot]","[Rose, Pear, Magnolia]","[Musk, Ambroxan, Dreamwood, Sandalwood]",2022.0


In [150]:
df["Perfumers"].dtype

dtype('O')

In [ ]:
def extract_years_from_description(df):
    pattern = r"was launched in (\d{4})"

    df["Launch_year"] = (
        df["Description"]
        .str.extract(pattern)[0]
        .astype("Int64")
    )

    return df

In [151]:
df = extract_years_from_description(df)
df.head(5)

,Name,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url,rating_value,rating_count,Top Notes,Middle Notes,Base Notes,launch_year
0,9am Afnan,women,3.73,174,"[citrus, musky, woody, aromatic, warm spicy, l...",[],9ambyAfnanis a fragrance for women. Top notes ...,https://www.fragrantica.com/perfume/Afnan/9am-...,3.73,174,"[Lemon, Mandarin Orange, Cardamom, Pink Pepper]","[Lavender, Green Apple, Orange Blossom, Rose]","[Musk, Moss, Cedar, Patchouli]",NaN
1,9am Dive Afnan,unisex,4.29,842,"[fruity, woody, green, warm spicy, aromatic, c...",[],9am DivebyAfnanis a Aromatic Aquatic fragrance...,https://www.fragrantica.com/perfume/Afnan/9am-...,4.29,842,"[Lemon, Mint, Black Currant, Pink Pepper]","[Apple, Incense, Cedar]","[Ginger, Sandalwood, Patchouli, Jasmine]",2022.0
2,9am pour Femme Afnan,women,4.00,68,"[fruity, musky, amber, citrus, powdery, sweet,...",[],9am pour FemmebyAfnanis a Amber fragrance for ...,https://www.fragrantica.com/perfume/Afnan/9am-...,4.00,68,"[Mandarin Orange, Grapefruit, Bergamot]","[Raspberry, Black Currant]","[Musk, Amber, Orange]",2022.0
3,9pm Afnan,men,4.50,"6,865","[vanilla, amber, warm spicy, cinnamon, sweet, ...",[],9pmbyAfnanis a Amber Vanilla fragrance for men...,https://www.fragrantica.com/perfume/Afnan/9pm-...,4.50,0,"[Apple, Cinnamon, Wild Lavender, Bergamot]","[Orange Blossom, Lily-of-the-Valley]","[Vanilla, Tonka Bean, Amber, Patchouli]",2020.0
4,9pm pour Femme Afnan,women,3.49,63,"[woody, aromatic, rose, fruity, powdery, viole...",[],9pm pour FemmebyAfnanis a Amber Floral fragran...,https://www.fragrantica.com/perfume/Afnan/9pm-...,3.49,63,"[Raspberry, Violet, Apple, Orange]","[Rose, Iris, Peony, Jasmine]","[Cypress, Pine, Cedar, Amber]",2022.0
